# Registros · Videoclub

Aplicación de gestión de un videoclub con socios, películas, ejemplares, alquileres, reservas e historial. Cada ejercicio amplía el programa anterior; aquí se construye por partes y se prueba con una sesión de ejemplo.

## 1. Socios y películas

Definir `lee_pelicula`, `contiene_pelicula_con_titulo`, `alta_pelicula` y `baja_pelicula`, evitando llamadas repetidas a la misma búsqueda (se busca una vez y se reutiliza el resultado).

In [1]:
from dataclasses import dataclass, field
from datetime import date, timedelta

@dataclass
class Socio:
    dni: str
    nombre: str
    historial: list = field(default_factory=list)

@dataclass
class Ejemplar:
    alquilado_a: str | None = None
    fecha_alquiler: date | None = None

@dataclass
class Pelicula:
    titulo: str
    generos: list
    dias_permitidos: int = 3
    ejemplares: list = field(default_factory=lambda: [Ejemplar()])
    reservas: list = field(default_factory=list)
    veces_alquilada: int = 0

socios, peliculas = {}, {}

def lee_pelicula():
    titulo = input("Título: ")
    generos = [g.strip() for g in input("Géneros (separados por comas): ").split(",")]
    dias = int(input("Días permitidos: "))
    n = int(input("Ejemplares: "))
    return Pelicula(titulo, generos, dias, [Ejemplar() for _ in range(n)])

def contiene_pelicula_con_titulo(titulo):
    return titulo in peliculas

def alta_pelicula(p):
    if contiene_pelicula_con_titulo(p.titulo):
        peliculas[p.titulo].ejemplares += p.ejemplares
    else:
        peliculas[p.titulo] = p

def baja_pelicula(titulo):
    p = peliculas.get(titulo)
    if p is None:
        return "No existe"
    if any(e.alquilado_a for e in p.ejemplares):
        return "No se puede dar de baja: tiene ejemplares alquilados"
    del peliculas[titulo]
    return "Baja correcta"

alta_pelicula(lee_pelicula())
for titulo, generos in (("Amélie", ["comedia", "romance"]), ("Alien", ["ciencia ficción", "terror"]), ("Up", ["animación", "comedia"])):
    alta_pelicula(Pelicula(titulo, generos))
for dni, nombre in (("1Z", "Ana"), ("2X", "Luis"), ("3Y", "Eva")):
    socios[dni] = Socio(dni, nombre)
print(list(peliculas), len(peliculas["Matrix"].ejemplares))

Título: Matrix
Géneros (separados por comas): ciencia ficción, acción
Días permitidos: 3
Ejemplares: 2
['Matrix', 'Amélie', 'Alien', 'Up'] 2


## 2. Alquilar y devolver

Alquilar una película a un socio (registrando fecha, contador y historial) y devolverla con el título y el DNI. Devolver indica los días de retraso según `dias_permitidos`; si la película no existe o no la tenía alquilada ese socio, avisar.

In [2]:
def disponibles(p):
    return [e for e in p.ejemplares if e.alquilado_a is None]

def alquila(titulo, dni, hoy):
    p = peliculas.get(titulo)
    if p is None:
        return "No existe"
    libres = disponibles(p)
    # los ejemplares libres quedan apartados para quien reservó antes
    reservas_vivas = [r for r in p.reservas if r[0] != dni]
    if len(libres) <= len(reservas_vivas):
        return "No disponible (puede reservarla)"
    libres[0].alquilado_a, libres[0].fecha_alquiler = dni, hoy
    p.reservas = [r for r in p.reservas if r[0] != dni]
    p.veces_alquilada += 1
    socios[dni].historial.append(titulo)
    return "Alquilada"

def devolver_pelicula(titulo, dni, hoy):
    p = peliculas.get(titulo)
    if p is None:
        return "No existe esa película"
    ejemplar = next((e for e in p.ejemplares if e.alquilado_a == dni), None)
    if ejemplar is None:
        return "Ese socio no la tenía alquilada"
    retraso = max(0, (hoy - ejemplar.fecha_alquiler).days - p.dias_permitidos)
    ejemplar.alquilado_a = ejemplar.fecha_alquiler = None
    for reserva in p.reservas:
        if reserva[1] is None:
            # empieza a contar la caducidad de la reserva
            reserva[1] = hoy
            break
    return f"Devuelta con {retraso} días de retraso" if retraso else "Devuelta a tiempo"

hoy = date(2026, 11, 1)
print(alquila("Alien", "1Z", hoy), alquila("Alien", "2X", hoy))
print(devolver_pelicula("Alien", "2X", hoy), devolver_pelicula("Alien", "1Z", hoy + timedelta(days=6)), devolver_pelicula("Psicosis", "1Z", hoy))

Alquilada No disponible (puede reservarla)
Ese socio no la tenía alquilada Devuelta con 3 días de retraso No existe esa película


## 3. Bajas protegidas y listados por género

No permitir dar de baja una película alquilada ni a un socio con películas alquiladas. Listar por género los títulos en orden alfabético, sin repetir títulos con varios ejemplares, indicando `disponible` si queda al menos un ejemplar libre. Una película puede tener varios géneros.

In [3]:
def socio_tiene_alquileres(dni):
    return any(e.alquilado_a == dni for p in peliculas.values() for e in p.ejemplares)

def baja_socio(dni):
    if socio_tiene_alquileres(dni):
        return "No se puede: tiene películas alquiladas"
    socios.pop(dni, None)
    return "Baja correcta"

def listado_completo_por_genero(genero):
    for p in sorted(peliculas.values(), key=lambda p: p.titulo):
        if genero in p.generos:
            print(f"{p.titulo:10} {'disponible' if disponibles(p) else 'no disponible'}")

print(alquila("Up", "3Y", hoy), baja_pelicula("Up"), baja_socio("3Y"), baja_socio("2X"))
listado_completo_por_genero("comedia")
listado_completo_por_genero("ciencia ficción")

Alquilada No se puede dar de baja: tiene ejemplares alquilados No se puede: tiene películas alquiladas Baja correcta
Amélie     disponible
Up         no disponible
Alien      disponible
Matrix     disponible


## 4. Reservas con caducidad

Si no hay ejemplares libres, un socio puede reservar la película (no un ejemplar). Con dos reservas pendientes, solo se alquila a otros cuando hay tres o más ejemplares libres. La reserva caduca si el socio no alquila la película en los dos días siguientes a quedar disponible.

In [4]:
def reserva(titulo, dni):
    peliculas[titulo].reservas.append([dni, None])

def caduca_reservas(hoy):
    for p in peliculas.values():
        p.reservas = [r for r in p.reservas if r[1] is None or (hoy - r[1]).days <= 2]

socios["4W"] = Socio("4W", "Iker")
print(alquila("Up", "1Z", hoy))
reserva("Up", "1Z")
print(devolver_pelicula("Up", "3Y", hoy + timedelta(days=1)))
print("Iker:", alquila("Up", "4W", hoy + timedelta(days=1)))
caduca_reservas(hoy + timedelta(days=5))
print("Iker tras caducar la reserva:", alquila("Up", "4W", hoy + timedelta(days=5)))

No disponible (puede reservarla)
Devuelta a tiempo
Iker: No disponible (puede reservarla)
Iker tras caducar la reserva: Alquilada


## 5. Más alquiladas, género favorito y consejos

Registrar cuántas veces se ha alquilado cada película y mostrar las 10 más alquiladas; consultar el género (o géneros) favorito de un socio según su historial; y aconsejarle películas de esos géneros realmente disponibles.

In [5]:
from collections import Counter

def mas_alquiladas(n=10):
    return [(p.titulo, p.veces_alquilada) for p in sorted(peliculas.values(), key=lambda p: -p.veces_alquilada)[:n]]

def generos_favoritos(dni):
    cuenta = Counter(g for titulo in socios[dni].historial if titulo in peliculas for g in peliculas[titulo].generos)
    maximo = max(cuenta.values(), default=0)
    return [g for g, n in cuenta.items() if n == maximo]

def aconseja(dni):
    favoritos = generos_favoritos(dni)
    return [p.titulo for p in peliculas.values()
            if set(p.generos) & set(favoritos) and len(disponibles(p)) > len(p.reservas) and p.titulo not in socios[dni].historial]

print(mas_alquiladas())
print(socios["1Z"].historial, generos_favoritos("1Z"), aconseja("1Z"))

[('Up', 2), ('Alien', 1), ('Matrix', 0), ('Amélie', 0)]
['Alien'] ['ciencia ficción', 'terror'] ['Matrix']
